# 条件熵、信息增益与树模型

## 1. 已经知道一些信息后，还剩多少不确定性？

上一篇我们把 entropy 理解为概率分布的平均信息量。现在进一步问：如果我们已经知道一些额外信息，原来的不确定性会下降多少？

例如，我们想预测广州某天是否下雨。如果什么都不知道，只看全年数据，降雨和不降雨都可能发生，天气有一定不确定性。但如果知道当天处在高温雨季、低温干季或过渡季，我们对是否下雨的判断会更有把握。这时，问题就从 entropy 变成 conditional entropy。

这和计量经济学里熟悉的条件期望、条件概率是一脉相承的。$E(Y|X)$ 讨论已知 $X$ 后 $Y$ 的平均水平；$P(Y=1|X)$ 讨论已知 $X$ 后事件发生概率；$H(Y|X)$ 讨论已知 $X$ 后，$Y$ 的概率分布还剩多少不确定性。


## 2. 从条件概率到条件熵

假设 $Y$ 是一个信息变量，例如气温区间、月份、行业类别或企业规模组。对于每个 $Y=y$，我们都可以观察 $X$ 的条件分布 $P(X|Y=y)$，并计算这个条件分布的 entropy：

$$
H(X|Y=y)=-\sum_x P(x|y)\log_2 P(x|y)
$$

但总体上，我们通常不只关心某一个条件状态，而是关心“平均而言，知道 $Y$ 之后，$X$ 还剩多少不确定性”。因此，要对所有 $y$ 做加权平均：

$$
H(X|Y)=-\sum_y P(y)\sum_x P(x|y)\log_2 P(x|y)
$$

这就是 conditional entropy。

::: {.callout-note}
### 条件熵的关键词是“剩余”

普通 entropy $H(X)$ 度量 $X$ 的概率分布本身有多不确定。条件熵 $H(X|Y)$ 则度量：在已经知道 $Y$ 的情况下，关于 $X$ 平均还剩多少不确定性。
:::


## 3. 广州天气的例子

令 $X=1$ 表示下雨，$X=0$ 表示不下雨。再令 $Y$ 表示气温区间：低温区、中温区和高温区。不同气温区间下，降雨概率不同，因此条件分布也不同。

![](./figs/entropy_cond_fig01_weather_groups.png){width="74%"}

在低温区，下雨概率很低，因此天气比较容易预测，条件 entropy 较低。在中温区，下雨和不下雨都比较可能，因此条件 entropy 较高。在高温区，下雨概率较高，但并非完全确定，因此 entropy 介于二者之间。

条件熵 $H(X|Y)$ 就是把这些局部不确定性按照 $P(Y)$ 做加权平均。


In [4]:
def entropy(prob):
    """离散概率分布的 Shannon entropy。"""
    prob = np.asarray(prob, dtype=float)
    prob = prob[prob > 0]
    return -np.sum(prob * np.log2(prob))

# 气温区间示例：X 为是否下雨，Y 为气温区间。
weather = pd.DataFrame({
    'group': ['低温区', '中温区', '高温区'],
    'P(Y)': [0.30, 0.40, 0.30],
    'P(rain|Y)': [0.05, 0.40, 0.75],
})
weather['P(no rain|Y)'] = 1 - weather['P(rain|Y)']
weather['H(X|Y=y)'] = weather.apply(lambda r: entropy([r['P(rain|Y)'], r['P(no rain|Y)']]), axis=1)

H_cond = np.sum(weather['P(Y)'] * weather['H(X|Y=y)'])
p_rain = np.sum(weather['P(Y)'] * weather['P(rain|Y)'])
H_total = entropy([p_rain, 1 - p_rain])
IG = H_total - H_cond
print(weather)
print('-' * 40)
print(f'H(X)   = {H_total:.3f}')
print(f'H(X|Y) = {H_cond:.3f}')
print(f'IG     = {IG:.3f}')


  group  P(Y)  P(rain|Y)  P(no rain|Y)  H(X|Y=y)
0   低温区   0.3       0.05          0.95  0.286397
1   中温区   0.4       0.40          0.60  0.970951
2   高温区   0.3       0.75          0.25  0.811278
----------------------------------------
H(X)   = 0.971
H(X|Y) = 0.718
IG     = 0.253


## 4. 信息增益：一个变量减少了多少不确定性？

知道 $Y$ 之前，我们对 $X$ 的不确定性是 $H(X)$。知道 $Y$ 之后，剩余不确定性是 $H(X|Y)$。两者之差就是 information gain：

$$
IG(X,Y)=H(X)-H(X|Y)
$$

它度量的是：变量 $Y$ 给我们提供了多少关于 $X$ 的信息。

![](./figs/entropy_cond_fig02_information_gain.png){width="70%"}

这和回归中的解释力有相似直觉。在线性回归中，加入解释变量后，残差平方和下降，说明解释变量提供了有用信息；在分类问题中，加入特征后，entropy 下降，说明特征减少了分类不确定性。但二者不能简单等同，因为 RSS 衡量的是数值误差，而 information gain 衡量的是概率分布不确定性的下降。


## 5. 决策树中的 split quality

决策树的一个核心问题是：在某个节点上，应该选择哪个变量、哪个切点来划分样本？对于分类树，一个好的 split 应该让子节点更纯。若父节点中 blue 和 green 混在一起，entropy 较高；分裂之后，如果左子节点几乎全是 blue，右子节点几乎全是 green，那么子节点的加权 entropy 会下降。

设父节点样本量为 $n$，分裂后有 $K$ 个子节点，第 $k$ 个子节点样本量为 $n_k$，则 split 后的加权 entropy 为：

$$
H_{split}=\sum_{k=1}^{K}\frac{n_k}{n}H(child_k)
$$

信息增益为：

$$
IG=H(parent)-\sum_{k=1}^{K}\frac{n_k}{n}H(child_k)
$$

决策树会在候选变量和候选切点中寻找能让 $IG$ 最大的 split。

![](./figs/entropy_cond_fig03_tree_split.png){width="74%"}


## 6. 为什么信息增益能指导树模型？

树模型的目标不是直接估计一个参数，而是逐步把样本空间切成若干区域，使每个区域内部的标签尽量一致。对于分类问题，区域内部越纯，预测越容易。

如果某个 split 几乎没有改变类别混合程度，那么 split 前后的 entropy 差不多，information gain 很小。这样的 split 对分类帮助不大。如果某个 split 让子节点中的类别明显分离，weighted child entropy 会显著下降，information gain 较大。这样的 split 才值得保留。

这也是为什么 entropy 和 Gini impurity 经常同时出现在分类树中。二者都是 impurity criterion，都试图衡量节点内部的类别混杂程度。Gini impurity 的计算更简单，entropy 的信息论解释更直接。在很多实际数据中，它们选择的 split 往往相近，但不是完全相同。

::: {.callout-tip}
### Random Forest 中 entropy 的位置

Random Forest 的重点是 ensemble：通过 bootstrap 抽样和随机特征选择生成许多棵树，再把这些树的预测结果汇总。Entropy 通常不是“森林层面”的指标，而是每棵分类树在节点分裂时可以使用的 criterion。也就是说，RF 是组合框架，entropy 是单棵树内部可使用的一把 split 尺子。
:::


## 7. 从条件熵到特征选择

在更一般的机器学习语境中，information gain 可以被理解为一种特征评价方式。如果某个变量 $Y$ 让 $H(X|Y)$ 明显低于 $H(X)$，说明它对预测 $X$ 有用；如果 $H(X|Y)$ 几乎等于 $H(X)$，说明知道 $Y$ 并没有减少我们对 $X$ 的不确定性。

这可以帮助读者把条件熵和熟悉的条件概率联系起来：

- 如果 $P(X|Y)$ 随 $Y$ 明显变化，$Y$ 通常能减少 $X$ 的不确定性。
- 如果 $P(X|Y)=P(X)$，则 $Y$ 对 $X$ 没有信息，$H(X|Y)=H(X)$。
- 如果 $Y$ 完全决定 $X$，则 $H(X|Y)=0$，因为知道 $Y$ 后 $X$ 不再有不确定性。

这种思路不仅适用于天气例子，也适用于金融分类问题。例如，若企业的财务杠杆、现金流、行业景气度能够明显改变违约概率，那么这些变量就会降低违约标签的条件熵。


## 8. 小结

这一篇的主线可以概括为：

- $H(X)$ 度量不知道任何额外信息时，$X$ 的概率分布有多不确定。
- $H(X|Y)$ 度量知道 $Y$ 后，关于 $X$ 平均还剩多少不确定性。
- $H(X)-H(X|Y)$ 是 information gain，衡量 $Y$ 减少了多少不确定性。
- 决策树用 information gain 评价 split quality，希望每次分裂都让子节点更纯。
- Random Forest 中的 entropy 通常作用于单棵分类树的节点分裂，而不是森林整体。

下一篇会从另一个方向展开：当模型给出预测概率后，如何衡量预测分布与真实结果之间的错配？这就进入 cross-entropy、KL divergence 和 entropy balancing。


## 参考文献

- Quinlan, J. R. (1986). Induction of decision trees. *Machine Learning*, 1, 81-106. [Link](https://doi.org/10.1007/BF00116251), [PDF](https://hunch.net/~coms-4771/quinlan.pdf), [Google](https://scholar.google.com/scholar?q=Induction+of+Decision+Trees+Quinlan+1986).
- Breiman, L. (2001). Random forests. *Machine Learning*, 45, 5-32. [Link](https://doi.org/10.1023/A:1010933404324), [PDF](https://www.stat.berkeley.edu/~breiman/randomforest2001.pdf), [Google](https://scholar.google.com/scholar?q=Random+Forests+Leo+Breiman+2001).
